In [ ]:
import os
import base64
from ollama import Client

# 1. Set your API key
OLLAMA_API_KEY = ""  # Or set in env var
client = Client(
    host="https://ollama.com",
    headers={"Authorization": "Bearer " + OLLAMA_API_KEY}
)

# 2. Load an image and encode it to base64
with open("test-data/AncientEgyptianArchitecture.webp", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')

# 3. Send request to a vision-capable cloud model
response = client.chat(
    model="qwen3-vl:235b-cloud",  # Use a fully vision-capable model
    messages=[{
        "role": "user",
        "content": "give metadata of this image in json format.",
        "images": [base64_image]   # Crucial: Include images as base64[reference:8]
    }]
)

# 4. Print the response
print(response["message"]["content"])

{
  "title": "Artistic Depiction of Ancient Egyptian Monuments at Sunset",
  "description": "A photorealistic digital artwork showcasing iconic ancient Egyptian structures, including pyramids, the Sphinx, colossal statues, and temples, set against a desert landscape with rocky mountains and a sunset sky.",
  "subject": "Ancient Egyptian architecture and historical monuments",
  "location": "Artistic composite inspired by Giza (pyramids, Sphinx) and Theban necropolis (rock-cut temples, e.g., Abu Simbel/Karnak)",
  "time_of_day": "Sunset",
  "color_palette": ["warm orange", "golden brown", "sandy beige", "deep blue", "soft pink", "dusty red"],
  "key_elements": [
    "Pyramids",
    "Great Sphinx",
    "Colossal seated statues (Abu Simbel-inspired)",
    "Columned temples (Karnak-style architecture)",
    "Desert landscape",
    "Rocky mountain formations",
    "Palm trees",
    "Small water pools",
    "Tiny human figures (scale reference)",
    "Sunset sky with scattered clouds"
  ],
 

In [2]:
# test_captioner.py

from __future__ import annotations

import base64
import json
import re
from pathlib import Path
from typing import Any

import structlog
from ollama import Client

logger = structlog.get_logger()


# ============================================================
# CONFIG
# ============================================================

OLLAMA_BASE_URL = "https://ollama.com"

# Recommended:
# qwen3-vl:235b-cloud
MODEL = "qwen3-vl:235b-cloud"

# Test image
IMAGE_PATH = "test-data/AncientEgyptianArchitecture.webp"


# ============================================================
# PROMPT
# ============================================================

_CAPTION_PROMPT = """
You are generating architectural retrieval metadata for a precedent image database.

Analyze the image and return ONLY valid JSON.

PRIMARY GOAL:
Generate dense architectural retrieval information suitable for:
- semantic search
- architectural precedent discovery
- architectural style retrieval
- material and composition filtering
- RAG systems

RULES:
- Describe only visible architectural evidence.
- Prefer observable features over interpretation.
- Avoid poetic language.
- Avoid conceptual architectural criticism.
- Use concise architectural terminology.
- Focus strongly on architectural style and visual characteristics.
- Building names should only appear if highly recognizable.
- Avoid redundancy.
- Prefer atomic descriptors.

Return concise, information-dense output.

Schema:
{
  "view_type": "",
  "architectural_style": [],
  "materials": [],
  "structure": [],
  "composition": [],
  "spatial_features": [],
  "lighting": [],
  "program_hints": [],
  "tags": [],
  "caption": "",
  "embedding_text": "",
  "raw_visual_description": "",
  "identity": {
      "building_name": null,
      "confidence": 0.0
  }
}

Field guidance:

architectural_style:
Examples:
- brutalist
- modernist
- postmodern
- vernacular
- contemporary
- gothic
- art deco
- bauhaus
- deconstructivist
- high-tech
- neo-futurist
- ancient egyptian
- classical
- neoclassical
- islamic
- traditional japanese

materials:
Examples:
- exposed concrete
- limestone
- sandstone
- brick
- timber
- weathered steel
- glass curtain wall
- stone facade

composition:
Examples:
- symmetry
- axial composition
- monumental massing
- repetitive bays
- vertical rhythm
- deep reveals
- layered facade
- stepped geometry

tags:
Dense retrieval-oriented architectural tags.
Maximum 20.
Prefer short atomic phrases.

caption:
One concise retrieval-oriented sentence.

embedding_text:
Compact semantic-search text under 80 words.

raw_visual_description:
Dense architectural prose describing visible features.
"""


# ============================================================
# NORMALIZATION
# ============================================================

NORMALIZATION_MAP = {
    # Concrete
    "raw concrete": "exposed concrete",
    "unfinished concrete": "exposed concrete",
    "fair-faced concrete": "exposed concrete",

    # Glass
    "glass facade": "glass curtain wall",
    "curtain glazing": "glass curtain wall",

    # Composition
    "repetitive facade": "repetitive bays",
    "recessed windows": "deep reveals",

    # Styles
    "modern": "modernist",
    "neo brutalist": "brutalist",
}


# ============================================================
# EMPTY RESULT TEMPLATE
# ============================================================

_EMPTY_RESULT: dict[str, Any] = {
    "view_type": "",
    "architectural_style": [],
    "materials": [],
    "structure": [],
    "composition": [],
    "spatial_features": [],
    "lighting": [],
    "program_hints": [],
    "tags": [],
    "caption": "",
    "embedding_text": "",
    "raw_visual_description": "",
    "identity": {
        "building_name": None,
        "confidence": 0.0,
    },
}


# ============================================================
# MAIN
# ============================================================

def main() -> None:

    image_path = Path(IMAGE_PATH)

    if not image_path.exists():
        raise FileNotFoundError(
            f"Image not found: {IMAGE_PATH}"
        )

    print("\n==================================================")
    print("ARCHITECTURAL IMAGE TEST")
    print("==================================================")

    print(f"\nMODEL: {MODEL}")
    print(f"IMAGE: {IMAGE_PATH}")

    # --------------------------------------------------------
    # LOAD IMAGE
    # --------------------------------------------------------

    with open(image_path, "rb") as image_file:
        image_b64 = base64.b64encode(
            image_file.read()
        ).decode("utf-8")

    # --------------------------------------------------------
    # CLIENT
    # --------------------------------------------------------
    client = Client(
    host="https://ollama.com",
    headers={"Authorization": "Bearer " + OLLAMA_API_KEY}
    )

    print("\nRunning Qwen3-VL inference...\n")

    # --------------------------------------------------------
    # INFERENCE
    # --------------------------------------------------------

    response = client.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": _CAPTION_PROMPT,
                "images": [image_b64],
            }
        ],
        options={
            "temperature": 0.1,
            "top_p": 0.9,
        },
        think=False,
        format="json",
    )

    raw = (
        response.message.content or ""
    ).strip()

    # --------------------------------------------------------
    # RAW RESPONSE
    # --------------------------------------------------------

    print("==================================================")
    print("RAW MODEL RESPONSE")
    print("==================================================\n")

    print(raw)

    # --------------------------------------------------------
    # PARSE
    # --------------------------------------------------------

    result = _parse_json_safe(raw)

    # --------------------------------------------------------
    # NORMALIZE
    # --------------------------------------------------------

    result = _normalize_result(result)

    result["method"] = f"ollama/{MODEL}"

    # --------------------------------------------------------
    # FINAL JSON
    # --------------------------------------------------------

    print("\n==================================================")
    print("NORMALIZED RESULT")
    print("==================================================\n")

    print(
        json.dumps(
            result,
            indent=2,
            ensure_ascii=False,
        )
    )

    # --------------------------------------------------------
    # QUICK SUMMARY
    # --------------------------------------------------------

    print("\n==================================================")
    print("SUMMARY")
    print("==================================================\n")

    print("VIEW TYPE:")
    print(result.get("view_type"))

    print("\nARCHITECTURAL STYLE:")
    print(result.get("architectural_style"))

    print("\nMATERIALS:")
    print(result.get("materials"))

    print("\nCOMPOSITION:")
    print(result.get("composition"))

    print("\nTAGS:")
    print(result.get("tags"))

    print("\nCAPTION:")
    print(result.get("caption"))

    print("\nEMBEDDING TEXT:")
    print(result.get("embedding_text"))

    print("\nIDENTITY:")
    print(result.get("identity"))

    print("\n==================================================\n")


# ============================================================
# NORMALIZATION
# ============================================================

def _normalize_result(
    result: dict[str, Any]
) -> dict[str, Any]:

    fields = [
        "architectural_style",
        "materials",
        "structure",
        "composition",
        "spatial_features",
        "lighting",
        "program_hints",
        "tags",
    ]

    for field in fields:

        values = result.get(field, [])

        if not isinstance(values, list):
            values = []

        normalized = []

        for value in values:

            if not value:
                continue

            value = (
                str(value)
                .strip()
                .lower()
            )

            value = NORMALIZATION_MAP.get(
                value,
                value,
            )

            normalized.append(value)

        # dedupe while preserving order
        result[field] = list(
            dict.fromkeys(normalized)
        )

    return result


# ============================================================
# JSON PARSER
# ============================================================

def _parse_json_safe(
    raw: str
) -> dict[str, Any]:

    raw = raw.strip()

    # Remove Qwen thinking blocks
    raw = re.sub(
        r"<think>.*?</think>",
        "",
        raw,
        flags=re.DOTALL,
    ).strip()

    # Remove markdown fences
    raw = re.sub(
        r"^```(?:json)?\s*",
        "",
        raw,
    )

    raw = re.sub(
        r"\s*```$",
        "",
        raw,
    )

    raw = raw.strip()

    # Attempt direct parse
    try:
        return json.loads(raw)

    except json.JSONDecodeError:
        pass

    # Attempt extraction of first JSON object
    match = re.search(
        r"\{[\s\S]*\}",
        raw,
    )

    if match:
        try:
            return json.loads(match.group())

        except json.JSONDecodeError:
            pass

    logger.warning(
        "caption_json_parse_failed",
        raw_preview=raw[:500],
    )

    return {
        **_EMPTY_RESULT,
        "parse_error": True,
        "raw_response": raw[:2000],
    }


# ============================================================
# ENTRY
# ============================================================

if __name__ == "__main__":
    main()


ARCHITECTURAL IMAGE TEST

MODEL: qwen3-vl:235b-cloud
IMAGE: test-data/AncientEgyptianArchitecture.webp

Running Qwen3-VL inference...

RAW MODEL RESPONSE

{
  "view_type": "panoramic",
  "architectural_style": ["ancient egyptian"],
  "materials": ["limestone", "sandstone"],
  "structure": ["monumental stone blocks", "columned portico", "pyramidal form"],
  "composition": ["axial composition", "symmetry", "monumental massing", "hierarchical spatial organization"],
  "spatial_features": ["open courtyards", "processional pathways", "stepped terraces"],
  "lighting": ["golden hour lighting", "long shadows"],
  "program_hints": ["funerary complex", "temple complex"],
  "tags": [
    "ancient egyptian",
    "pyramids",
    "sphinx",
    "limestone",
    "sandstone",
    "columned portico",
    "funerary complex",
    "monumental stone",
    "axial symmetry",
    "desert landscape",
    "hieroglyphic carvings",
    "processional way",
    "temple complex",
    "desert architecture",
    "his

In [ ]:
import os
import base64
from ollama import Client

# 1. Set your API key
OLLAMA_API_KEY = ""  # Or set in env var
client = Client(
    host="https://ollama.com",
    headers={"Authorization": "Bearer " + OLLAMA_API_KEY}
)

# 2. Load an image and encode it to base64
with open("test-data/AncientEgyptianArchitecture.webp", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')

# 3. Send request to a vision-capable cloud model
response = client.chat(
    model="gemma4:31b-cloud",  # Use a fully vision-capable model
    messages=[{
        "role": "user",
        "content": "give metadata of this image in json format.",
        "images": [base64_image]   # Crucial: Include images as base64[reference:8]
    }]
)

# 4. Print the response
print(response["message"]["content"])

Since I am an AI, I cannot access the original binary EXIF data (such as camera model, GPS, or timestamp) embedded in the image file. However, I can provide the **descriptive metadata** based on the image content.

```json
{
  "image_description": "A composite digital artwork depicting an idealized ancient Egyptian landscape featuring the Sphinx, pyramids, and a large temple complex set against red rock cliffs during sunset.",
  "visual_elements": {
    "foreground": "Ancient stone ruins, small turquoise pools of water, and scattered palm trees.",
    "midground": "The Great Sphinx, a colonnaded temple complex, and small human figures for scale.",
    "background": "Large pyramids to the left and towering reddish-brown sandstone mountains/cliffs to the right.",
    "lighting": "Golden hour/Sunset with a warm orange and yellow glow on the horizon and a blue-to-orange gradient sky."
  },
  "style": "Digital illustration / AI-generated",
  "color_palette": [
    "#E2A76F (Sandy Gold)",
  

: 